# Oil Spill Semantic Segmentation — Final Integration Demo
### One SAR image · Ground Truth · Four trained models · Shared metrics

**Team experiments**
1. U-Net + Cross-Entropy (Member 1)
2. U-Net + Cross-Entropy & Dice (Member 2)
3. DeepLabV3+ ResNet34 Frozen Encoder (Member 3)
4. DeepLabV3+ ResNet34 Fine-tuned Encoder (Member 4)

**What this notebook does:** loads **existing trained best checkpoints** from the [team GitHub release](https://github.com/Nada-Alhady/Oil-Spill-Semantic-Segmentation/releases/tag/v1.0.0), uses each architecture's correct input normalization, and compares **real predictions on the same held-out SAR image**. It does **not train** anything.

#### Before pressing Run All on Kaggle
- **Add Input** → attach the **oil-spill** Kaggle dataset with `train/images`, `train/labels`, `test/images`, and `test/labels` (1002 official training pairs and 110 official test pairs).
- **Settings → Internet → On** is needed to install `segmentation-models-pytorch` and download the public GitHub release. If downloading the release is blocked, **upload `OilSpill_Final_Submission_v1.zip` as a Kaggle input** instead; the notebook finds it automatically.
- **Settings → Accelerator → GPU T4** (recommended). CPU can also work, but slower.
- Choose a test-image index in the Configuration cell. `AUTO_PICK_OIL=True` finds a sample with visible oil-spill pixels.

**Evaluation note:** All models use the original input pipeline appropriate for their weights (U-Net: RGB 0–1; DeepLab: ImageNet mean/std). The optional unified test evaluation uses one common ground-truth mask conversion and excludes class ID 255 (unknown pixels). Cross-architecture training/validation splits still differed; do not claim training protocols were identical.

## 1 — Install inference library (no training)

In [ ]:
%pip -q install segmentation-models-pytorch==0.5.0

## 2 — Imports & configuration

In [ ]:
from pathlib import Path
import os, io, csv, json, gc, zipfile, shutil
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from PIL import Image
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
import segmentation_models_pytorch as smp

IMG_SIZE = 256
NUM_CLASSES = 5
CLASS_NAMES = ["Sea", "Oil spill", "Look-alike", "Ships", "Land"]
SOURCE_MASK_RGB = [(0,0,0), (0,255,255), (255,0,0), (153,76,0), (0,153,0)]
DISPLAY_RGB = np.array([(40,120,181), (230,57,70), (244,197,66), (168,85,247), (66,165,95)], dtype=np.uint8)
IGNORE_ID = 255
MEAN = np.array([.485,.456,.406], dtype=np.float32).reshape(3,1,1)
STD = np.array([.229,.224,.225], dtype=np.float32).reshape(3,1,1)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
INPUT_ROOT = Path('/kaggle/input') if Path('/kaggle/input').exists() else Path.cwd()
WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
OUT = WORK / 'Final_Integration_Demo_Outputs'
OUT.mkdir(parents=True, exist_ok=True)
WEIGHTS_DIR = WORK / 'OilSpill_Demo_Weights'
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

# Change these only if you want a different demo picture or the full 110-image run:
TEST_INDEX = 0
AUTO_PICK_OIL = True
RUN_FULL_TEST = False  # Change to True for one uniform evaluation across 110 test images

REPO_RELEASE = 'https://github.com/Nada-Alhady/Oil-Spill-Semantic-Segmentation/releases/download/v1.0.0/OilSpill_Final_Submission_v1.zip'
EXPECTED = {
 'Member1': 'Member1_best_model.pt',
 'Member2': 'Member2_best_model.pt',
 'Member3': 'Member3_best_checkpoint.pth',
 'Member4': 'Member4_best_checkpoint.pth',
}
print('PyTorch:', torch.__version__, '| SMP:', smp.__version__)
print('Inference device:', DEVICE, '| Output folder:', OUT)

## 3 — Locate pretrained weights (Kaggle input or GitHub Release)

Only the **four best model weights** are extracted. We do not use `last_checkpoint` or any optimizer state to train. This may download the public 422 MB release once; subsequent notebook cells use local files.

In [ ]:
def find_exact_weight(filename):
    # Prefer extracted files, otherwise recognize existing Kaggle dataset attachments.
    matches = [p for root in (WEIGHTS_DIR, INPUT_ROOT) for p in root.rglob(filename) if p.is_file()]
    return matches[0] if matches else None


def extract_best_weights(zip_path):
    print('Opening:', zip_path)
    with zipfile.ZipFile(zip_path) as z:
        for name, filename in EXPECTED.items():
            dest = WEIGHTS_DIR / filename
            if dest.exists() and dest.stat().st_size > 100_000:
                continue
            matches = [n for n in z.namelist() if n.endswith('/Models/' + filename) or n == 'Models/' + filename]
            if len(matches) != 1:
                raise FileNotFoundError(f'Expected {filename} once in ZIP; found {matches}. Use the final team release ZIP.')
            with z.open(matches[0]) as src, dest.open('wb') as dst:
                shutil.copyfileobj(src, dst, length=8*1024*1024)
            print('Extracted:', filename, 'MB:', round(dest.stat().st_size/1024**2, 1))

missing = [k for k,v in EXPECTED.items() if find_exact_weight(v) is None]
if missing:
    zips = [p for root in (INPUT_ROOT, WORK) for p in root.rglob('OilSpill_Final_Submission_v1.zip') if p.is_file()]
    if zips:
        extract_best_weights(zips[0])
    else:
        release_zip = WORK / 'OilSpill_Final_Submission_v1.zip'
        print('Downloading PUBLIC GitHub release once. GPU is NOT used for downloading.')
        try:
            with requests.get(REPO_RELEASE, stream=True, timeout=(30,180)) as response:
                response.raise_for_status()
                with release_zip.open('wb') as f:
                    for chunk in response.iter_content(chunk_size=4*1024*1024):
                        if chunk:
                            f.write(chunk)
            extract_best_weights(release_zip)
        except Exception as exc:
            raise RuntimeError('Release download failed. In Kaggle turn Internet ON, OR attach OilSpill_Final_Submission_v1.zip using Add Input. Error: ' + str(exc)) from exc

MODEL_FILES = {k: find_exact_weight(v) for k,v in EXPECTED.items()}
assert all(MODEL_FILES.values()), f'Missing weights: {MODEL_FILES}'
for k,p in MODEL_FILES.items():
    print(k, p, f'{p.stat().st_size/1024**2:.1f} MiB')

## 4 — Find the original held-out test images and RGB masks

In [ ]:
def discover_dataset():
    roots = sorted({p.parent for p in INPUT_ROOT.rglob('test')
                    if p.is_dir() and (p/'images').is_dir() and (p/'labels').is_dir()
                    and (p.parent/'train'/'images').is_dir()}, key=str)
    if len(roots) != 1:
        raise FileNotFoundError(f'Expected ONE oil-spill dataset with test/images and test/labels; found: {roots}. Kaggle > Add Input > oil-spill.')
    return roots[0]

DATA = discover_dataset()
images_dir = DATA / 'test' / 'images'
masks_dir = DATA / 'test' / 'labels'
TEST_PAIRS = []
for im_path in sorted(images_dir.iterdir(), key=lambda p:p.name):
    if im_path.suffix.lower() not in ('.jpg','.jpeg','.png','.tif','.tiff'):
        continue
    mask_path = masks_dir / f'{im_path.stem}.png'
    assert mask_path.exists(), f'Missing mask for: {im_path.name}'
    TEST_PAIRS.append((im_path, mask_path))
assert len(TEST_PAIRS) == 110, f'Expected 110 official test images, got {len(TEST_PAIRS)}'
print('Dataset:', DATA)
print('Official test image/mask pairs:', len(TEST_PAIRS))

## 5 — Reconstruct the ORIGINAL architectures (from the saved team sources)

U-Net matches the custom four-level model (base channels = 32, 5 output channels). DeepLabV3+ matches SMP 0.5.0 with a ResNet34 encoder, but uses `encoder_weights=None` at inference because the trained state dict will replace all weights.

In [ ]:
class ConvBlock(nn.Sequential):
    def __init__(self, in_channels, out_channels):
        super().__init__(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True)
        )

class UNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=5, base_channels=32):
        super().__init__()
        c1,c2,c3,c4,c5 = (base_channels*k for k in (1,2,4,8,16))
        self.enc1 = ConvBlock(in_channels,c1)
        self.enc2 = ConvBlock(c1,c2)
        self.enc3 = ConvBlock(c2,c3)
        self.enc4 = ConvBlock(c3,c4)
        self.bottleneck = ConvBlock(c4,c5)
        self.pool = nn.MaxPool2d(2)
        self.up4 = nn.ConvTranspose2d(c5,c4,2,2)
        self.dec4 = ConvBlock(c4*2,c4)
        self.up3 = nn.ConvTranspose2d(c4,c3,2,2)
        self.dec3 = ConvBlock(c3*2,c3)
        self.up2 = nn.ConvTranspose2d(c3,c2,2,2)
        self.dec2 = ConvBlock(c2*2,c2)
        self.up1 = nn.ConvTranspose2d(c2,c1,2,2)
        self.dec1 = ConvBlock(c1*2,c1)
        self.classifier = nn.Conv2d(c1,num_classes,1)

    @staticmethod
    def _merge(up,skip):
        if up.shape[-2:] != skip.shape[-2:]:
            up = F.interpolate(up, size=skip.shape[-2:], mode='bilinear', align_corners=False)
        return torch.cat((skip,up), dim=1)

    def forward(self,x):
        x1 = self.enc1(x)
        x2 = self.enc2(self.pool(x1))
        x3 = self.enc3(self.pool(x2))
        x4 = self.enc4(self.pool(x3))
        center = self.bottleneck(self.pool(x4))
        x = self.dec4(self._merge(self.up4(center), x4))
        x = self.dec3(self._merge(self.up3(x), x3))
        x = self.dec2(self._merge(self.up2(x), x2))
        x = self.dec1(self._merge(self.up1(x), x1))
        return self.classifier(x)

MODELS = {}
for name in EXPECTED:
    if name in ('Member1','Member2'):
        net = UNet(num_classes=5)
    else:
        net = smp.DeepLabV3Plus(
            encoder_name='resnet34', encoder_weights=None,
            in_channels=3, classes=5, activation=None
        )
    # Checkpoints are from our own team; only load checkpoints you trust.
    ckpt = torch.load(MODEL_FILES[name], map_location='cpu', weights_only=False)
    assert 'model_state_dict' in ckpt, f'Invalid checkpoint for {name}'
    net.load_state_dict(ckpt['model_state_dict'], strict=True)
    net.to(DEVICE).eval()
    MODELS[name] = net
    print(name, 'loaded | trained epoch:', ckpt.get('epoch', '?'), '| parameter count:', sum(p.numel() for p in net.parameters()))
    del ckpt
    gc.collect()
assert len(MODELS) == 4
print('All four real pretrained checkpoints loaded successfully.')

## 6 — Shared ground truth and architecture-specific image preprocessing

The unknown-mask-color rule is the same for the **visual demo and optional unified evaluation**: map unknown colors to 255 and ignore those pixels. Ground truth is mapped **before nearest-neighbor resizing** as in the DeepLab test evaluation.

In [ ]:
def load_sample(pair):
    image_path, mask_path = pair
    with Image.open(image_path) as img:
        rgb = np.asarray(img.convert('RGB').resize((IMG_SIZE,IMG_SIZE), Image.Resampling.BILINEAR), dtype=np.uint8).copy()
    with Image.open(mask_path) as mask_image:
        mask_rgb = np.asarray(mask_image.convert('RGB'), dtype=np.uint8)
    labels = np.full(mask_rgb.shape[:2], IGNORE_ID, dtype=np.uint8)
    for i,color in enumerate(SOURCE_MASK_RGB):
        labels[np.all(mask_rgb == np.asarray(color), axis=-1)] = i
    truth = np.asarray(Image.fromarray(labels).resize((IMG_SIZE,IMG_SIZE), Image.Resampling.NEAREST)).copy()
    normal = rgb.astype(np.float32).transpose(2,0,1)/255.0
    unet_tensor = torch.from_numpy(normal.copy()).unsqueeze(0).to(DEVICE)
    deeplab_tensor = torch.from_numpy(((normal-MEAN)/STD).copy()).unsqueeze(0).to(DEVICE)
    return rgb, truth, unet_tensor, deeplab_tensor

@torch.inference_mode()
def run_models(unet_tensor, deeplab_tensor):
    predictions = {}
    for name, net in MODELS.items():
        x = unet_tensor if name in ('Member1','Member2') else deeplab_tensor
        logits = net(x)
        assert logits.shape == (1,NUM_CLASSES,IMG_SIZE,IMG_SIZE), (name,logits.shape)
        predictions[name] = logits.argmax(1)[0].cpu().numpy().astype(np.uint8)
    return predictions

def rgb_visual(mask):
    safe = mask.clip(0,4)
    arr = DISPLAY_RGB[safe].copy()
    arr[mask == IGNORE_ID] = (180,180,180)
    return arr

def image_metrics(truth, prediction):
    ok = truth != IGNORE_ID
    t = truth[ok].astype(np.int64)
    p = prediction[ok].astype(np.int64)
    cm = np.bincount(NUM_CLASSES*t+p, minlength=NUM_CLASSES*NUM_CLASSES).reshape(NUM_CLASSES,NUM_CLASSES)
    true_n=cm.sum(axis=1); pred_n=cm.sum(axis=0); inter=cm.diagonal()
    union=true_n+pred_n-inter
    iou=np.divide(inter, union, out=np.full(NUM_CLASSES,np.nan), where=union>0)
    den=true_n+pred_n
    dice=np.divide(2*inter, den, out=np.full(NUM_CLASSES,np.nan), where=den>0)
    return {'mIoU':float(np.nanmean(iou)), 'Mean Dice':float(np.nanmean(dice)),
            'Pixel Accuracy':float(inter.sum()/cm.sum()), 'Oil Spill IoU':float(iou[1]) if np.isfinite(iou[1]) else np.nan,
            'confusion':cm, 'per_class_iou':iou, 'per_class_dice':dice}

## 7 — REAL live demo: one image, Ground Truth, and 4 predictions

In [ ]:
assert 0 <= TEST_INDEX < len(TEST_PAIRS)
selected_idx = TEST_INDEX
if AUTO_PICK_OIL:
    for i,pair in enumerate(TEST_PAIRS):
        _, truth_candidate, _, _ = load_sample(pair)
        if (truth_candidate == 1).sum() > 20:
            selected_idx = i
            break

image_path,mask_path = TEST_PAIRS[selected_idx]
rgb, truth, u_input, d_input = load_sample((image_path,mask_path))
predictions = run_models(u_input, d_input)
fig, axes = plt.subplots(2,3, figsize=(16,11))
items = [
 ('Original SAR image',rgb),('Ground truth',rgb_visual(truth)),
 ('Member 1 — U-Net CE',rgb_visual(predictions['Member1'])),
 ('Member 2 — U-Net CE+Dice',rgb_visual(predictions['Member2'])),
 ('Member 3 — DeepLab Frozen',rgb_visual(predictions['Member3'])),
 ('Member 4 — DeepLab Fine-tuned',rgb_visual(predictions['Member4']))
]
for ax,(title,array) in zip(axes.flat,items):
    ax.imshow(array, interpolation='nearest'); ax.set_title(title,fontsize=12);ax.axis('off')
fig.legend([Patch(facecolor=DISPLAY_RGB[i]/255) for i in range(NUM_CLASSES)],CLASS_NAMES,
           loc='lower center',ncol=5,bbox_to_anchor=(.5,.01))
fig.suptitle(f'Actual predictions on SAME test image: {image_path.name}',fontsize=16)
fig.tight_layout(rect=(0,.055,1,.96))
figure_path = OUT / 'Same_Image_Four_Model_Predictions.png'
fig.savefig(figure_path,dpi=170,bbox_inches='tight')
plt.show()

rows=[]
for name,pred in predictions.items():
    met = image_metrics(truth,pred)
    rows.append({'Model':name,**{k:met[k] for k in ['mIoU','Mean Dice','Pixel Accuracy','Oil Spill IoU']}})
single_df=pd.DataFrame(rows).set_index('Model')
display(single_df.round(4))
single_df.to_csv(OUT/'Selected_Image_Comparison.csv')
print('Saved:',figure_path)

## 8 — Optional: unified evaluation on **all 110 official test images**

Change `RUN_FULL_TEST = True` in the configuration cell **if the doctor wants a fresh apples-to-apples test comparison**. This does inference only, not training. It uses exactly the same RGB-mask conversion, 255 ignore policy, and confusion-matrix aggregation for all four models. The original reported test metrics may differ slightly from these rerun values.

In [ ]:
if RUN_FULL_TEST:
    confusions={k:np.zeros((NUM_CLASSES,NUM_CLASSES),dtype=np.int64) for k in MODELS}
    total_ignored=0
    for pair in tqdm(TEST_PAIRS,desc='Unified test inference'):
        rgb,mask,u,d=load_sample(pair)
        preds=run_models(u,d)
        total_ignored += int((mask==IGNORE_ID).sum())
        for k,p in preds.items():
            confusions[k]+=image_metrics(mask,p)['confusion']
    all_rows=[];class_rows=[]
    for k,cm in confusions.items():
        inter=cm.diagonal().astype(float)
        true_n=cm.sum(1).astype(float)
        pred_n=cm.sum(0).astype(float)
        union=true_n+pred_n-inter
        iou=np.divide(inter,union,out=np.full(NUM_CLASSES,np.nan),where=union>0)
        dice=np.divide(2*inter,true_n+pred_n,out=np.full(NUM_CLASSES,np.nan),where=(true_n+pred_n)>0)
        all_rows.append({'Model':k,'Test mIoU':np.nanmean(iou),'Mean Dice':np.nanmean(dice),
                         'Pixel Accuracy':inter.sum()/cm.sum(),'Oil Spill IoU':iou[1]})
        for i in range(NUM_CLASSES):
            class_rows.append({'Model':k,'Class':CLASS_NAMES[i],'IoU':iou[i],'Dice':dice[i]})
    all_df=pd.DataFrame(all_rows).set_index('Model')
    display(all_df.round(4))
    all_df.to_csv(OUT/'Unified_110_Test_Comparison.csv')
    pd.DataFrame(class_rows).to_csv(OUT/'Unified_110_Per_Class.csv',index=False)
    print('Ignored unknown-color pixels (same mask for every model):',total_ignored)
    fig,axes=plt.subplots(1,2,figsize=(12,5))
    all_df[['Test mIoU','Mean Dice']].plot.bar(ax=axes[0],rot=12,title='Unified test: mIoU / Mean Dice')
    all_df[['Oil Spill IoU']].plot.bar(ax=axes[1],rot=12,title='Unified test: Oil Spill IoU')
    for ax in axes:ax.set_ylim(0,1);ax.set_ylabel('Score')
    fig.tight_layout()
    fig.savefig(OUT/'Unified_110_Comparison_Charts.png',dpi=170)
    plt.show()
else:
    print('Full test evaluation skipped (RUN_FULL_TEST=False). The same-image live demo above is complete.')

## 9 — Save outputs for the final presentation and GitHub

In [ ]:
ZIP_OUT = WORK / 'Final_Integration_Demo_Results.zip'
with zipfile.ZipFile(ZIP_OUT,'w',zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob('*'):
        if p.is_file():
            z.write(p,p.relative_to(OUT.parent))
print('DONE — demo assets saved to:', ZIP_OUT)
for p in sorted(OUT.iterdir()): print(' -',p.name)
try:
    from IPython.display import FileLink,display
    display(FileLink(str(ZIP_OUT)))
except Exception:
    pass

---
### What to show the doctor
1. **This notebook’s live picture:** 1 original SAR image + its Ground Truth + 4 **real predictions** of the *same sample*.
2. **Single-image metric table:** scores from the actual four predictions.
3. **Original training/results report:** results at the project [GitHub repository](https://github.com/Nada-Alhady/Oil-Spill-Semantic-Segmentation) and [release](https://github.com/Nada-Alhady/Oil-Spill-Semantic-Segmentation/releases/tag/v1.0.0).
4. Optional: rerun full 110-image unified test with `RUN_FULL_TEST=True` for standardized numbers.

**Important:** This file was generated from the team's saved source and checkpoint structure. It must be run **once on Kaggle with the dataset attached** before the presentation. Do not say a live four-model demo has been verified until the Kaggle run succeeds.